# Project 2: finding scratches on wafers

**DBA5106, Project 2, Version 2 (with data).** Dataset: MixedWM38 mixed-type wafer maps (Kaggle `co1d7era/mixedtype-wafer-defect-datasets`, usability 7.5).

## The decision this model supports
A wafer map shows which dies passed and which failed electrical test. A **scratch** shows up as a thin, often curved line of failing dies. It comes from mechanical contact: a handling robot, a wafer carrier, or a polishing step. Two things make it worth catching:

- **It repeats.** The handler that scratched this wafer will scratch the next ones until someone fixes it.
- **It hides risk.** Dies next to a scratch can pass test and still fail early in the field, so a scratched wafer needs its neighbouring dies screened before it ships.

In this dataset a wafer can carry several defect patterns at once (for example a centre blob, an edge ring and a scratch), so the scratch has to be found among other defects.

The model reads a wafer map and predicts **P(the wafer has a scratch)**. If it says "scratch", the handling path is held for inspection and the wafer goes to neighbour-die screening.

## Why the evaluation loss is not the misclassification rate
- **False negative** (a scratched wafer not flagged): the handler keeps scratching wafers until a routine check finds it, and this wafer's at-risk dies ship unscreened.
- **False positive** (a clean wafer flagged): one handler inspection that finds nothing, and one wafer screened unnecessarily.

A miss carries both a repeating cost and a reliability cost; a false alarm carries a one-off inspection. We assume a miss costs **10 times** a false alarm. The dataset has no cost data, so this ratio is our assumption, and section 8 shows how the decision changes for ratios from 1 to 20.

$$\text{evaluation loss} = \frac{10 \cdot FN + 1 \cdot FP}{n} \quad \text{(cost per wafer, in units of one unnecessary inspection)}$$

## How each requirement is met
| Requirement | Where |
|---|---|
| Classification; evaluation loss deviates from misclassification, from the context | Above, and section 3 |
| At least one glassbox and one blackbox | Section 4: L1-penalised linear model; gradient-boosted trees |
| Each model in two versions: L2 loss vs another surrogate loss, tuned separately | Section 4: squared error vs binomial deviance (log loss), four separate searches |
| Compare all of them on validation AUC (not train loss) and decide | Section 5 |
| Threshold that minimises the evaluation loss, for the chosen model | Section 6 |
| Is the model calibrated? If not, improve it | Section 7 |
| Tree SHAP on a tree-based blackbox, local and global | Section 9 |
| Good habits: few explorations, no validation overfitting | Sections 2, 4 and 10 |

## Design choices
1. **Features built for the question.** A scratch is a line, so seven of the sixteen features measure line-shaped evidence. The other nine describe the background: how many dies fail, where, and how clustered.
2. **No feature says where the line is.** In these maps scratches tend to sit in the same part of the wafer. A real scratch can be anywhere, so position is left out on purpose.
3. **Two things removed before modelling**, with the evidence in section 1: exact duplicate maps, and the die count.
4. **Three separate jobs for the data.** Hyperparameters are tuned by cross-validation inside the training split. The validation split compares the four tuned models and sets the threshold. The test split is used once, at the end.
5. **Uncertainty on every comparison**, from a bootstrap over wafers.


## 1. Setup and data

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from IPython.display import display, Markdown

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, confusion_matrix
from sklearn.base import clone
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 42
C_FN, C_FP = 10.0, 1.0           # evaluation loss: a missed scratch costs 10 unnecessary inspections
N_BOOT = 2000                    # bootstrap resamples of wafers

DATA = "data/mixedwm38_scratch_features.csv"
MAPS = "data/mixedwm38_scratch_maps.npz"
REPORT = "data/mixedwm38_cleaning_report.json"
FIG = "figures_scratch"; os.makedirs(FIG, exist_ok=True)

# Plot style: glassbox / no scratch = blue, blackbox / Scratch = orange; solid = log loss, dashed = L2
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "axes.axisbelow": True, "legend.fontsize": 8, "figure.dpi": 110,
    "lines.linewidth": 2,
})
def savefig(name):
    plt.savefig(os.path.join(FIG, name), dpi=200, bbox_inches="tight")

df = pd.read_csv(DATA)
BACKGROUND = ["fail_rate", "center_fail_rate", "middle_fail_rate", "edge_fail_rate", "outer_ring_fail_rate",
              "n_fail_clusters", "largest_cluster_share", "largest_cluster_radius", "angular_concentration"]
LINE = ["thin_fail_rate", "longest_thin_length", "n_long_thin", "line_peak_contrast",
        "longest_line_length", "n_line_segments", "line_score_top20"]
FEATURES = BACKGROUND + LINE
TARGET = "scratch"
X, y = df[FEATURES], df[TARGET]
print(f"{len(df):,} wafers, {len(FEATURES)} features ({len(BACKGROUND)} background, {len(LINE)} line), scratch rate {y.mean():.1%}")
df.head()

36,991 wafers, 16 features (9 background, 7 line), scratch rate 48.7%


,wafer_id,fail_rate,center_fail_rate,middle_fail_rate,edge_fail_rate,outer_ring_fail_rate,n_fail_clusters,largest_cluster_share,largest_cluster_radius,angular_concentration,...,longest_thin_length,n_long_thin,line_peak_contrast,longest_line_length,n_line_segments,line_score_top20,pattern,n_patterns,scratch,split
0,0,0.300825,0.524324,0.220443,0.323308,0.509709,106,0.129549,0.884098,0.154518,...,0.256460,3,0.033978,0.219265,17,0.631566,Center + Edge-Loc + Scratch,3,1,train
1,1,0.300825,0.875676,0.210591,0.269737,0.368932,121,0.098981,0.222773,0.093790,...,0.230926,1,0.059481,0.108786,12,0.594517,Center + Edge-Loc + Scratch,3,1,validation
2,2,0.327996,0.789189,0.274631,0.288534,0.419903,92,0.099951,0.246756,0.034598,...,0.382714,2,0.085075,0.231569,17,0.649098,Center + Edge-Loc + Scratch,3,1,train
3,3,0.277050,0.329730,0.184729,0.338346,0.514563,122,0.078117,0.888960,0.124899,...,0.300396,4,0.035845,0.190375,13,0.631385,Center + Edge-Loc + Scratch,3,1,test
4,4,0.248423,0.459459,0.250000,0.210526,0.266990,126,0.044153,0.205165,0.109107,...,0.317504,2,0.034861,0.138675,9,0.600505,Center + Edge-Loc + Scratch,3,1,validation


**The data.** MixedWM38 has 38,015 wafer maps, all 52 × 52 dies. Each is labelled with every pattern it contains: 8 basic patterns (Center, Donut, Edge-Loc, Edge-Ring, Loc, Near-full, Scratch, Random) in 38 combinations. The target is 1 if the label includes Scratch. `build_features_mixedwm38.py` cleans the maps, computes the features and fixes the split.

**Background features (9)**

| Feature | Meaning |
|---|---|
| `fail_rate` | Share of dies that fail |
| `center_fail_rate`, `middle_fail_rate`, `edge_fail_rate`, `outer_ring_fail_rate` | Fail rate in the centre (radius < 0.3), middle (0.3–0.7), edge (≥ 0.7) and outermost ring (≥ 0.9); the radius is 1 at the outermost die |
| `n_fail_clusters` | Number of connected groups of failing dies |
| `largest_cluster_share` | Size of the largest group, as a share of all dies |
| `largest_cluster_radius` | Mean radius of the largest group (0 = centre, 1 = edge) |
| `angular_concentration` | 0 if failures are spread all around the wafer, 1 if they sit on one side |

**Line features (7)**

| Feature | Meaning |
|---|---|
| `thin_fail_rate` | Share of dies that fail and belong to a thin structure (a line or a speck, not a blob) |
| `longest_thin_length` | Length of the longest thin structure, as a share of the wafer's width |
| `n_long_thin` | Number of thin structures at least 10 dies long |
| `line_peak_contrast` | How much the strongest straight line across the wafer stands out from a typical direction |
| `longest_line_length` | Length of the longest run of dies whose failing neighbours line up in one direction |
| `n_line_segments` | Number of such runs |
| `line_score_top20` | Average "lined-up" score of the 20 most line-like failing dies |

The line features were designed on the training split only, in three rounds (thin structures; local direction; both together with the background features).

In [2]:
# What was removed when the table was built, and why
rep = json.load(open(REPORT))
display(pd.Series({
    "maps in the source file": rep["maps_read"],
    "exact duplicate maps dropped": rep["duplicate_maps_dropped"],
    "share of the dropped duplicates that are scratched": f"{rep['share_of_dropped_duplicates_with_scratch']:.1%}",
    "maps kept": rep["maps_kept"],
    "die count, scratched wafers (min to max)": f"{rep['die_count_scratch']['min']:,} to {rep['die_count_scratch']['max']:,}",
    "die count, unscratched wafers (min to max)": f"{rep['die_count_no_scratch']['min']:,} to {rep['die_count_no_scratch']['max']:,}",
    "unscratched wafers whose die count no scratched wafer has": f"{rep['share_of_no_scratch_wafers_outside_the_scratch_die_count_range']:.1%}",
}).to_frame("value"))
print("Duplicates: copies of one map in both training and validation would inflate every score, and they were almost all scratched.")
print("Die count: every map is the same 52 x 52 grid, so the count has no physical link to a scratch, yet some counts occur only on")
print("unscratched wafers. A model could use that to rule scratches out. It is not written to the table.")

# Data checks: all of these use features only, no labels
checks = pd.Series({
    "missing cells": int(df[FEATURES + [TARGET]].isna().sum().sum()),
    "duplicate feature rows": int(df.duplicated(subset=FEATURES).sum()),
    "constant features": int((df[FEATURES].nunique() <= 1).sum()),
    "wafers with more than one pattern": int((df["n_patterns"] > 1).sum()),
    "scratched wafers that also carry another pattern": int(((df["scratch"] == 1) & (df["n_patterns"] > 1)).sum()),
})
display(checks.to_frame("value"))
display(df[FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].round(3))

,value
maps in the source file,38015
exact duplicate maps dropped,1024
share of the dropped duplicates that are scratched,97.7%
maps kept,36991
"die count, scratched wafers (min to max)","2,061 to 2,065"
"die count, unscratched wafers (min to max)","1,843 to 2,234"
unscratched wafers whose die count no scratched wafer has,5.2%


Duplicates: copies of one map in both training and validation would inflate every score, and they were almost all scratched.
Die count: every map is the same 52 x 52 grid, so the count has no physical link to a scratch, yet some counts occur only on
unscratched wafers. A model could use that to rule scratches out. It is not written to the table.


,value
missing cells,0
duplicate feature rows,0
constant features,0
wafers with more than one pattern,29000
scratched wafers that also carry another pattern,17000


,mean,std,min,50%,max
fail_rate,0.319,0.085,0.010,0.316,0.950
center_fail_rate,0.422,0.232,0.000,0.384,1.000
middle_fail_rate,0.289,0.116,0.006,0.252,1.000
edge_fail_rate,0.325,0.100,0.008,0.322,0.899
outer_ring_fail_rate,0.449,0.213,0.002,0.439,0.894
n_fail_clusters,102.699,30.012,1.000,105.000,181.000
largest_cluster_share,0.154,0.110,0.000,0.129,0.950
largest_cluster_radius,0.648,0.235,0.045,0.714,0.998
angular_concentration,0.115,0.068,0.000,0.105,0.566
thin_fail_rate,0.170,0.032,0.000,0.178,0.237


## 2. Splits: train / validation / test
The dataset has no lot or time information, so the split is stratified random: 60 / 20 / 20. It is fixed in the data file, so every step uses the same partition. Exact duplicate maps were removed before splitting, so no identical wafer sits in two splits.

- **Train:** fit the models; tune hyperparameters by 5-fold cross-validation.
- **Validation:** compare the four tuned models on AUC, choose one, set its threshold.
- **Test:** touched once, in section 10.

**Disclosure.** An earlier version of this analysis, with a different feature set, was scored on this same test split. Its aggregate test numbers were seen. No test wafer was inspected, and the changes since then (the line features and the search grids) were designed on the training split.

In [3]:
train_idx = np.flatnonzero(df["split"] == "train"); val_idx = np.flatnonzero(df["split"] == "validation"); test_idx = np.flatnonzero(df["split"] == "test")
assert len(train_idx) + len(val_idx) + len(test_idx) == len(df)

X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], df["wafer_id"].iloc[train_idx]
X_va, y_va, g_va = X.iloc[val_idx], y.iloc[val_idx], df["wafer_id"].iloc[val_idx]
X_te, y_te, g_te = X.iloc[test_idx], y.iloc[test_idx], df["wafer_id"].iloc[test_idx]
display(pd.DataFrame({"wafers": [len(train_idx), len(val_idx), len(test_idx)],
                      "scratched": [int(y_tr.sum()), int(y_va.sum()), int(y_te.sum())],
                      "scratch rate": [f"{v:.1%}" for v in (y_tr.mean(), y_va.mean(), y_te.mean())]},
                     index=["train", "validation", "test"]))

,wafers,scratched,scratch rate
train,22195,10800,48.7%
validation,7398,3600,48.7%
test,7398,3600,48.7%


### A look at the training data
Only the training split is plotted, so nothing here leaks from validation or test.

In [4]:
fig, axes = plt.subplots(4, 4, figsize=(13, 9)); axes = axes.ravel()
for ax, c in zip(axes, FEATURES):
    v = X_tr[c]; hi = v.quantile(0.99) if c in ("n_fail_clusters", "n_line_segments", "n_long_thin") else v.max()
    bins = np.linspace(v.min(), hi, 31)
    ax.hist(v[y_tr == 0], bins=bins, density=True, histtype="step", lw=1.8, color=BLUE, label="No scratch")
    ax.hist(v[y_tr == 1], bins=bins, density=True, histtype="step", lw=1.8, color=ORANGE, label="Scratch")
    ax.set_title(c, fontsize=9); ax.set_yticks([])
axes[0].legend()
fig.suptitle("Feature distributions by class (training split)", fontweight="bold"); plt.tight_layout(); savefig("01_features_by_class.png"); plt.show()

corr = X_tr.corr()
hi_pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(corr.loc[a, b]) >= 0.85]
print("feature pairs with |r| ≥ 0.85 (training split):", hi_pairs)
auc1 = pd.Series({c: roc_auc_score(y_tr, X_tr[c]) for c in FEATURES}).map(lambda a: max(a, 1 - a)).sort_values(ascending=False)
print("single-feature AUC on the training split (0.5 = no signal):")
print(", ".join(f"{c} {v:.2f}" for c, v in auc1.items()))

feature pairs with |r| ≥ 0.85 (training split): [('fail_rate', 'largest_cluster_share', np.float64(0.87)), ('edge_fail_rate', 'outer_ring_fail_rate', np.float64(0.87)), ('outer_ring_fail_rate', 'n_line_segments', np.float64(0.91))]
single-feature AUC on the training split (0.5 = no signal):
longest_thin_length 0.73, n_long_thin 0.68, thin_fail_rate 0.66, n_line_segments 0.64, longest_line_length 0.64, line_score_top20 0.63, line_peak_contrast 0.63, edge_fail_rate 0.58, outer_ring_fail_rate 0.56, fail_rate 0.55, largest_cluster_share 0.55, n_fail_clusters 0.54, middle_fail_rate 0.54, largest_cluster_radius 0.54, center_fail_rate 0.51, angular_concentration 0.50


Several features move together (the zone fail rates with the overall fail rate; the counts of line segments with the fail rates). That matters twice later: an L1 penalty suits the linear glassbox, because it can drop redundant features, and SHAP credit can be shared arbitrarily between correlated features (section 9).

## 3. The evaluation loss
A function of the decision threshold: flag a wafer as scratched when its score is at least `t`.

In [5]:
def eval_loss(y_true, score, t, c_fn=C_FN, c_fp=C_FP):
    """Cost per wafer when wafers with score >= t are flagged as Scratch."""
    y_true = np.asarray(y_true); flag = np.asarray(score) >= t
    fn = np.sum((y_true == 1) & ~flag); fp = np.sum((y_true == 0) & flag)
    return (c_fn * fn + c_fp * fp) / len(y_true)

def best_threshold(y_true, score, c_fn=C_FN, c_fp=C_FP):
    """Exact minimiser over all thresholds that change a decision. Returns (threshold, cost)."""
    y_true = np.asarray(y_true); score = np.asarray(score)
    order = np.argsort(-score); s, yy = score[order], y_true[order]
    tp = np.cumsum(yy); fp = np.cumsum(1 - yy)                # flag the k highest scores
    cost = np.r_[c_fn * yy.sum(), c_fn * (yy.sum() - tp) + c_fp * fp] / len(yy)
    last = np.r_[True, np.r_[s[1:] != s[:-1], True]]          # only cut between distinct scores
    k = np.flatnonzero(last)[np.argmin(cost[last])]
    t = np.inf if k == 0 else (s[k - 1] if k == len(s) else (s[k - 1] + s[k]) / 2)
    return float(t), float(cost[k])

T_THEORY = C_FP / (C_FP + C_FN)
base = y_va.mean()
print(f"If the model's probabilities are calibrated, the cost-minimising threshold is C_FP / (C_FP + C_FN) = {T_THEORY:.3f}.")
print(f"Reference policies on validation: flag every wafer = {C_FP * (1 - base):.3f} per wafer; "
      f"flag none = {C_FN * base:.3f} per wafer.")

If the model's probabilities are calibrated, the cost-minimising threshold is C_FP / (C_FP + C_FN) = 0.091.
Reference policies on validation: flag every wafer = 0.513 per wafer; flag none = 4.866 per wafer.


## 4. Four models: glassbox and blackbox, each with L2 loss and with log loss
| | L2 loss (squared error on the 0/1 label) | Surrogate: binomial deviance (log loss) |
|---|---|---|
| **Glassbox:** L1-penalised linear model on standardised features | LASSO regression | L1 logistic regression |
| **Blackbox:** gradient-boosted trees | boosting with squared error | boosting with log loss |

**Why these two.** The line features are meant to turn "is there a line?" into numbers a simple model can add up, so a linear model is a fair glassbox to try, and L1 keeps it sparse. Boosted trees are the blackbox because scratch evidence should interact with the background: the same line score means more on a quiet wafer than on a noisy one.

Inside each row the model structure and the search grid are the same and **only the training loss changes**.

**Tuning.** Each of the four is tuned separately by 5-fold cross-validated AUC inside the training split. The grids are small and chosen in advance: 7 penalty strengths for the glassbox; tree depth (3, 5, 8) × number of trees (200, 500, 1,000) for the blackbox. That's 32 configurations in total. The validation split plays no part in tuning.

In [6]:
def score_of(model, X_):
    """One score per wafer: P(scratch) for log-loss models, the fitted value for L2 models."""
    return model.predict_proba(X_)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_)

def auc_scorer(model, X_, y_):
    return roc_auc_score(y_, score_of(model, X_))

SPECS = {
    ("glassbox", "L2"): (make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
                         {"lasso__alpha": [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]}),
    ("glassbox", "log loss"): (make_pipeline(StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, random_state=SEED)),
                               {"logisticregression__C": [0.003, 0.01, 0.03, 0.1, 1.0, 10.0, 100.0]}),
    ("blackbox", "L2"): (HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.1, random_state=SEED),
                         {"max_depth": [3, 5, 8], "max_iter": [200, 500, 1000]}),
    ("blackbox", "log loss"): (HistGradientBoostingClassifier(learning_rate=0.1, random_state=SEED),
                               {"max_depth": [3, 5, 8], "max_iter": [200, 500, 1000]}),
}
inner_cv = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y_tr))

MODELS, SEARCH, rows = {}, {}, []
for key, (est, grid) in SPECS.items():
    gs = GridSearchCV(est, grid, scoring=auc_scorer, cv=inner_cv, n_jobs=-1, refit=True).fit(X_tr, y_tr)
    MODELS[key] = gs.best_estimator_; SEARCH[key] = gs
    rows.append({"model": key[0], "train loss": key[1], "configurations tried": len(gs.cv_results_["params"]),
                 "chosen": ", ".join(f"{k.split('__')[-1]}={v}" for k, v in gs.best_params_.items()),
                 "CV AUC (train split)": round(gs.best_score_, 4)})
tuning = pd.DataFrame(rows); display(tuning)
print("total configurations tried:", int(tuning["configurations tried"].sum()))

,model,train loss,configurations tried,chosen,CV AUC (train split)
0,glassbox,L2,7,alpha=0.0001,0.8857
1,glassbox,log loss,7,C=100.0,0.9099
2,blackbox,L2,9,"max_depth=8, max_iter=500",0.9542
3,blackbox,log loss,9,"max_depth=8, max_iter=200",0.9590


total configurations tried: 32


### How much does the glassbox penalty matter?
The L1 penalty is what keeps a linear model readable: a stronger penalty leaves fewer features. This table shows every penalty value tried for the L1 logistic model, with its cross-validated AUC on the training split and the number of features it keeps. It uses the training split only.

In [7]:
gs = SEARCH[("glassbox", "log loss")]
rows = []
for params, mean, sd in zip(gs.cv_results_["params"], gs.cv_results_["mean_test_score"], gs.cv_results_["std_test_score"]):
    m = clone(SPECS[("glassbox", "log loss")][0]).set_params(**params).fit(X_tr, y_tr)
    kept = [f for f, c in zip(FEATURES, m[-1].coef_.ravel()) if c != 0]
    rows.append({"C (larger = weaker penalty)": params["logisticregression__C"], "CV AUC": round(mean, 4), "sd over folds": round(sd, 4),
                 "features kept": len(kept), "dropped": ", ".join(f for f in FEATURES if f not in kept) or "none"})
display(pd.DataFrame(rows))

,C (larger = weaker penalty),CV AUC,sd over folds,features kept,dropped
0,0.003,0.8610,0.0050,10,"fail_rate, center_fail_rate, middle_fail_rate,..."
1,0.010,0.8940,0.0039,12,"middle_fail_rate, largest_cluster_radius, angu..."
2,0.030,0.8968,0.0038,15,fail_rate
3,0.100,0.8973,0.0037,15,fail_rate
4,1.000,0.9092,0.0047,16,none
5,10.000,0.9099,0.0051,16,none
6,100.000,0.9099,0.0051,16,none


## 5. Compare on validation AUC and decide
AUC is computed on the validation split, which no model has seen. The intervals come from a bootstrap that resamples wafers 2,000 times, and differences between models are computed on the same resamples (paired).

**Decision rule, fixed before looking at the results:**
1. *Within a model type (L2 vs log loss):* take the higher validation AUC. If the difference isn't distinguishable from zero, take the log-loss version, because its output is a probability.
2. *Between model types:* an engineer has to act on a flag by holding a handler, so a model they can read is worth something. Choose the blackbox only if its validation AUC is distinguishably higher **and** that lowers the evaluation loss by at least 10%. Otherwise choose the glassbox.

In [8]:
S_va = {k: score_of(m, X_va) for k, m in MODELS.items()}
S_te = {k: score_of(m, X_te) for k, m in MODELS.items()}    # computed now, not looked at until section 10

def wafer_bootstrap(groups, n_boot=N_BOOT, seed=SEED):
    """Row indices for n_boot resamples of wafers, with replacement."""
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        yield rng.integers(0, len(groups), len(groups))

def boot_stats(y_true, groups, fns):
    """Bootstrap each statistic in fns (name -> function of row indices) on the same resamples."""
    y_true = np.asarray(y_true); out = {k: [] for k in fns}
    for idx in wafer_bootstrap(groups):
        if y_true[idx].min() == y_true[idx].max(): continue
        for k, f in fns.items(): out[k].append(f(idx))
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in out.items()}

yv = np.asarray(y_va)
ci = boot_stats(yv, g_va, {k: (lambda idx, s=S_va[k]: roc_auc_score(yv[idx], s[idx])) for k in MODELS})
auc_tab = pd.DataFrame([{"model": k[0], "train loss": k[1], "validation AUC": roc_auc_score(yv, S_va[k]),
                         "95% CI low": ci[k][0], "95% CI high": ci[k][1]} for k in MODELS])
display(auc_tab.round(4))

def auc_diff(a, b):
    d = roc_auc_score(yv, S_va[a]) - roc_auc_score(yv, S_va[b])
    lo, hi = boot_stats(yv, g_va, {"d": lambda idx: roc_auc_score(yv[idx], S_va[a][idx]) - roc_auc_score(yv[idx], S_va[b][idx])})["d"]
    return d, lo, hi

# Rule 1: within each family
best = {}
for fam in ("glassbox", "blackbox"):
    d, lo, hi = auc_diff((fam, "log loss"), (fam, "L2"))
    best[fam] = (fam, "L2") if hi < 0 else (fam, "log loss")
    print(f"{fam}: log loss − L2 validation AUC = {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f}) -> keep {best[fam][1]}")

# Rule 2: between families
d, lo, hi = auc_diff(best["blackbox"], best["glassbox"])
cost_g = best_threshold(yv, S_va[best["glassbox"]])[1]; cost_b = best_threshold(yv, S_va[best["blackbox"]])[1]
saving = 1 - cost_b / cost_g
print(f"\nblackbox − glassbox validation AUC = {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f})")
print(f"evaluation loss at each model's best threshold: glassbox {cost_g:.4f}, blackbox {cost_b:.4f} per wafer "
      f"-> the blackbox lowers it by {saving:.1%}")
CHOSEN = best["blackbox"] if (lo > 0 and saving >= 0.10) else best["glassbox"]
OTHER = best["glassbox"] if CHOSEN[0] == "blackbox" else best["blackbox"]
display(Markdown(f"**Decision: {CHOSEN[0]}, trained with {CHOSEN[1]}.**"))

,model,train loss,validation AUC,95% CI low,95% CI high
0,glassbox,L2,0.8850,0.8778,0.8921
1,glassbox,log loss,0.9088,0.9024,0.9154
2,blackbox,L2,0.9578,0.9537,0.9616
3,blackbox,log loss,0.9631,0.9593,0.9666


glassbox: log loss − L2 validation AUC = +0.0238 (95% CI +0.0199 to +0.0279) -> keep log loss


blackbox: log loss − L2 validation AUC = +0.0053 (95% CI +0.0037 to +0.0071) -> keep log loss



blackbox − glassbox validation AUC = +0.0543 (95% CI +0.0488 to +0.0598)
evaluation loss at each model's best threshold: glassbox 0.4059, blackbox 0.2499 per wafer -> the blackbox lowers it by 38.4%


**Decision: blackbox, trained with log loss.**

In [9]:
STYLE = {("glassbox", "L2"): (BLUE, "--"), ("glassbox", "log loss"): (BLUE, "-"),
         ("blackbox", "L2"): (ORANGE, "--"), ("blackbox", "log loss"): (ORANGE, "-")}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for k, s in S_va.items():
    fpr, tpr, _ = roc_curve(yv, s)
    for ax in axes: ax.plot(fpr, tpr, color=STYLE[k][0], ls=STYLE[k][1], lw=1.8, label=f"{k[0]}, {k[1]} (AUC {roc_auc_score(yv, s):.3f})")
axes[0].plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
axes[0].set_title("ROC on validation"); axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate"); axes[0].legend(loc="lower right")
axes[1].set_xlim(0, 0.3); axes[1].set_ylim(0.7, 1.0); axes[1].set_title("Zoom: the top-left corner"); axes[1].set_xlabel("false positive rate")
plt.tight_layout(); savefig("02_roc_validation.png"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 2.4))
for i, r in auc_tab.iloc[::-1].reset_index(drop=True).iterrows():
    k = (r["model"], r["train loss"])
    ax.plot([r["95% CI low"], r["95% CI high"]], [i, i], color=STYLE[k][0], lw=2, ls=STYLE[k][1])
    ax.plot(r["validation AUC"], i, "o", ms=8, color=STYLE[k][0], mec="#fcfcfb", mew=2)
    ax.text(r["95% CI high"] + 0.0008, i, f"{r['validation AUC']:.3f}", va="center", fontsize=8, color=INK2)
ax.set_yticks(range(4)); ax.set_yticklabels([f"{r['model']}, {r['train loss']}" for _, r in auc_tab.iloc[::-1].iterrows()])
ax.set_xlabel("validation AUC with 95% bootstrap interval"); ax.grid(axis="y", visible=False); ax.set_title("Four tuned models on validation")
plt.tight_layout(); savefig("03_auc_intervals.png"); plt.show()

### All four models, on more than one metric
The decision above uses validation AUC, as the brief requires. This table adds other views of the same four models. **Training AUC is shown only to make the brief's point:** it flatters the flexible model and says nothing about new wafers, which is why models are compared on validation.

In [10]:
from sklearn.metrics import average_precision_score, accuracy_score
ytr = np.asarray(y_tr); rows = []
for k, m in MODELS.items():
    s = S_va[k]; p = np.clip(s, 0, 1)
    rows.append({"model": k[0], "train loss": k[1],
                 "train AUC": roc_auc_score(ytr, score_of(m, X_tr)), "CV AUC (train)": SEARCH[k].best_score_,
                 "validation AUC": roc_auc_score(yv, s), "train − validation": roc_auc_score(ytr, score_of(m, X_tr)) - roc_auc_score(yv, s),
                 "average precision": average_precision_score(yv, s), "Brier": brier_score_loss(yv, p),
                 "accuracy at 0.5": accuracy_score(yv, s >= 0.5), "eval loss at 0.5": eval_loss(yv, s, 0.5),
                 "eval loss at best threshold": best_threshold(yv, s)[1]})
display(pd.DataFrame(rows).round(4))

,model,train loss,train AUC,CV AUC (train),validation AUC,train − validation,average precision,Brier,accuracy at 0.5,eval loss at 0.5,eval loss at best threshold
0,glassbox,L2,0.8860,0.8857,0.8850,0.0009,0.8785,0.1443,0.8087,1.1049,0.4496
1,glassbox,log loss,0.9102,0.9099,0.9088,0.0014,0.9059,0.1209,0.8273,0.9489,0.4059
2,blackbox,L2,0.9840,0.9542,0.9578,0.0262,0.9570,0.0833,0.8855,0.6206,0.2691
3,blackbox,log loss,0.9885,0.9590,0.9631,0.0254,0.9632,0.0764,0.8906,0.6130,0.2499


## 6. Threshold for the chosen model
For the chosen model we take the threshold that minimises the evaluation loss on validation, and compare it with the default 0.5.

In [11]:
s_va = S_va[CHOSEN]
T_STAR, cost_star = best_threshold(yv, s_va)
cost_05 = eval_loss(yv, s_va, 0.5)
ts = np.linspace(0.005, 0.995, 199)
curve = np.array([eval_loss(yv, s_va, t) for t in ts])
curve_mis = np.array([eval_loss(yv, s_va, t, 1, 1) for t in ts])
t_mis = best_threshold(yv, s_va, 1, 1)[0]

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(ts, curve, color=ORANGE, label=f"evaluation loss (FN = {C_FN:g}, FP = {C_FP:g})")
ax.plot(ts, curve_mis, color=BLUE, label="misclassification rate (FN = FP = 1)")
ax.axvline(T_STAR, color=INK2, lw=1, ls="--"); ax.text(T_STAR, curve.max() * 0.97, f" best: {T_STAR:.3f}", fontsize=8, color=INK2, va="top")
ax.axvline(0.5, color=MUTED, lw=1, ls=":"); ax.text(0.5, curve.max() * 0.97, " default 0.5", fontsize=8, color=MUTED, va="top")
ax.set_xlabel(f"threshold on the {CHOSEN[0]} score"); ax.set_ylabel("cost per wafer"); ax.set_title("Evaluation loss vs threshold (validation)"); ax.legend(loc="upper center")
plt.tight_layout(); savefig("04_threshold.png"); plt.show()

flag = s_va >= T_STAR
acc_star, acc_05 = (flag == yv).mean(), ((s_va >= 0.5) == yv).mean()
tn, fp, fn, tp = confusion_matrix(yv, flag).ravel()
print(f"best threshold = {T_STAR:.3f}: cost {cost_star:.4f} per wafer (FN {fn}, FP {fp}; recall {tp / (tp + fn):.1%}, flags {flag.mean():.1%} of wafers)")
tn5, fp5, fn5, tp5 = confusion_matrix(yv, s_va >= 0.5).ravel()
print(f"threshold 0.5:           cost {cost_05:.4f} per wafer (FN {fn5}, FP {fp5}; recall {tp5 / (tp5 + fn5):.1%})")
print(f"-> tuning the threshold lowers the evaluation loss by {1 - cost_star / cost_05:.1%}. "
      f"The threshold that minimises plain misclassification would be {t_mis:.3f}.")
print(f"accuracy: {acc_05:.1%} at 0.5, {acc_star:.1%} at the best threshold")
t_lo, t_hi = np.percentile([best_threshold(yv[i], s_va[i])[0] for i in wafer_bootstrap(g_va, 500)], [2.5, 97.5])
print(f"bootstrap 95% range of the best threshold: {t_lo:.3f} to {t_hi:.3f} (theory for a calibrated model: {T_THEORY:.3f})")

best threshold = 0.117: cost 0.2499 per wafer (FN 51, FP 1339; recall 98.6%, flags 66.1% of wafers)
threshold 0.5:           cost 0.6130 per wafer (FN 414, FP 395; recall 88.5%)
-> tuning the threshold lowers the evaluation loss by 59.2%. The threshold that minimises plain misclassification would be 0.480.
accuracy: 89.1% at 0.5, 81.2% at the best threshold
bootstrap 95% range of the best threshold: 0.074 to 0.168 (theory for a calibrated model: 0.091)


## 7. Calibration
A model is calibrated if, among wafers given a score near 0.3, about 30% are scratched. We measure it on validation with a reliability diagram, the expected calibration error (ECE: the average gap between predicted and observed rates, over 10 bins) and the Brier score.

**The fix.** We fit a calibrator on **out-of-fold training scores** (5 folds, same hyperparameters), then apply it to validation. Fitting the calibrator on validation would reuse the data that also sets the threshold. Two calibrators are tried: Platt scaling (a one-feature logistic regression, two parameters) and isotonic regression (a monotone step function).

In [12]:
def ece(y_true, p, bins=10):
    y_true = np.asarray(y_true); p = np.clip(np.asarray(p, float), 0, 1)
    b = np.clip((p * bins).astype(int), 0, bins - 1)
    return float(sum((b == i).mean() * abs(y_true[b == i].mean() - p[b == i].mean()) for i in range(bins) if (b == i).any()))

def reliability(y_true, p, bins=10):
    y_true = np.asarray(y_true); p = np.clip(np.asarray(p, float), 0, 1)
    q = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))               # equal-count bins
    b = np.clip(np.digitize(p, q[1:-1]), 0, len(q) - 2)
    return (np.array([p[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]),
            np.array([y_true[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]))

# Out-of-fold training scores for every model, with its tuned hyperparameters
OOF = {}
for k, m in MODELS.items():
    method = "predict_proba" if hasattr(m, "predict_proba") else "predict"
    o = cross_val_predict(clone(m), X_tr, y_tr, cv=inner_cv, method=method, n_jobs=-1)
    OOF[k] = o[:, 1] if o.ndim == 2 else o

def to_feature(k, s):
    """Platt input: the logit for probability models, the raw score for L2 models."""
    s = np.asarray(s, float)
    if k[1] == "L2": return s.reshape(-1, 1)
    s = np.clip(s, 1e-6, 1 - 1e-6); return np.log(s / (1 - s)).reshape(-1, 1)

CAL = {}
for k in MODELS:
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(to_feature(k, OOF[k]), y_tr)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(OOF[k], y_tr)
    CAL[k] = {"Platt": (lambda s, k=k, m=platt: m.predict_proba(to_feature(k, s))[:, 1]),
              "isotonic": (lambda s, m=iso: m.predict(np.asarray(s, float)))}

rows = []
for k in MODELS:
    raw = S_va[k]
    row = {"model": k[0], "train loss": k[1], "scores outside [0, 1]": f"{np.mean((raw < 0) | (raw > 1)):.1%}",
           "ECE raw": ece(yv, raw), "ECE Platt": ece(yv, CAL[k]["Platt"](raw)), "ECE isotonic": ece(yv, CAL[k]["isotonic"](raw)),
           "Brier raw": brier_score_loss(yv, np.clip(raw, 0, 1)), "Brier Platt": brier_score_loss(yv, CAL[k]["Platt"](raw)),
           "Brier isotonic": brier_score_loss(yv, CAL[k]["isotonic"](raw))}
    rows.append(row)
cal_tab = pd.DataFrame(rows); display(cal_tab.round(4))

def ece_ci(p):
    return np.percentile([ece(yv[i], p[i]) for i in wafer_bootstrap(g_va, 500)], [2.5, 97.5])
raw = S_va[CHOSEN]
for name, p in [("raw", raw), ("Platt", CAL[CHOSEN]["Platt"](raw)), ("isotonic", CAL[CHOSEN]["isotonic"](raw))]:
    lo, hi = ece_ci(p); print(f"chosen model, {name:8s}: ECE {ece(yv, p):.4f} (95% CI {lo:.4f} to {hi:.4f})")

,model,train loss,"scores outside [0, 1]",ECE raw,ECE Platt,ECE isotonic,Brier raw,Brier Platt,Brier isotonic
0,glassbox,L2,10.1%,0.0767,0.0168,0.0125,0.1443,0.1362,0.1361
1,glassbox,log loss,0.0%,0.0140,0.0142,0.0072,0.1209,0.1209,0.1210
2,blackbox,L2,24.3%,0.0348,0.0173,0.0076,0.0833,0.0813,0.0813
3,blackbox,log loss,0.0%,0.0114,0.0086,0.0087,0.0764,0.0763,0.0765


chosen model, raw     : ECE 0.0114 (95% CI 0.0094 to 0.0192)


chosen model, Platt   : ECE 0.0086 (95% CI 0.0070 to 0.0163)


chosen model, isotonic: ECE 0.0087 (95% CI 0.0073 to 0.0164)


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]; ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
for k in MODELS:
    mp, ob = reliability(yv, S_va[k]); ax.plot(mp, ob, marker="o", ms=4, lw=1.6, color=STYLE[k][0], ls=STYLE[k][1], label=f"{k[0]}, {k[1]}")
ax.set_title("Before calibration: all four models"); ax.set_xlabel("mean predicted probability (clipped to 0–1)"); ax.set_ylabel("observed scratch rate"); ax.legend()
ax = axes[1]; ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
col = STYLE[CHOSEN][0]
for name, p, ls, mk in [("raw", raw, "-", "o"), ("Platt", CAL[CHOSEN]["Platt"](raw), "--", "s"), ("isotonic", CAL[CHOSEN]["isotonic"](raw), ":", "^")]:
    mp, ob = reliability(yv, p); ax.plot(mp, ob, marker=mk, ms=5, lw=1.6, color=col if name == "raw" else INK2, ls=ls, label=f"{name} (ECE {ece(yv, p):.3f})")
ax.set_title(f"Chosen model ({CHOSEN[0]}, {CHOSEN[1]}): before and after"); ax.set_xlabel("mean predicted probability"); ax.legend()
plt.tight_layout(); savefig("05_calibration.png"); plt.show()

# Pick the calibrator by validation Brier score (a proper scoring rule); keep raw if neither helps
briers = {"raw": brier_score_loss(yv, np.clip(raw, 0, 1)), "Platt": brier_score_loss(yv, CAL[CHOSEN]["Platt"](raw)),
          "isotonic": brier_score_loss(yv, CAL[CHOSEN]["isotonic"](raw))}
CALIB = min(briers, key=briers.get)
calibrate = (lambda s: np.clip(np.asarray(s, float), 0, 1)) if CALIB == "raw" else CAL[CHOSEN][CALIB]
p_va = calibrate(raw)
T_CAL, cost_cal = best_threshold(yv, p_va)
print(f"calibrator with the lowest validation Brier score: {CALIB}  {({k: round(v, 4) for k, v in briers.items()})}")
print(f"AUC before {roc_auc_score(yv, raw):.4f}, after {roc_auc_score(yv, p_va):.4f} (a monotone map can only change AUC through ties)")
print(f"best threshold on calibrated probabilities = {T_CAL:.3f} (cost {cost_cal:.4f}); theory says {T_THEORY:.3f}, "
      f"which costs {eval_loss(yv, p_va, T_THEORY):.4f} on validation")

calibrator with the lowest validation Brier score: Platt  {'raw': 0.0764, 'Platt': 0.0763, 'isotonic': 0.0765}
AUC before 0.9631, after 0.9631 (a monotone map can only change AUC through ties)
best threshold on calibrated probabilities = 0.105 (cost 0.2499); theory says 0.091, which costs 0.2595 on validation


**Why this matters for the threshold.** If the probabilities are calibrated, we don't need to search for a threshold at all: decision theory gives it as C_FP / (C_FP + C_FN) = 1/11 ≈ 0.091, and it updates by formula when the costs change. The last line above checks how close the searched threshold on calibrated probabilities is to that value, and what using the formula would cost.

**The final decision rule** is: calibrate the chosen model's score, then hold the handler when the calibrated probability is at least the threshold found on validation.

## 8. How sensitive is the decision to the cost ratio?
The 10:1 ratio is our assumption. Here the ratio runs from 1 (plain misclassification) to 20. For each ratio we show the best threshold and the resulting cost, for the chosen model and for the best model of the other type.

In [14]:
ratios = np.array([1, 2, 3, 5, 8, 10, 15, 20])
p_other = CAL[OTHER][CALIB](S_va[OTHER]) if CALIB != "raw" else np.clip(S_va[OTHER], 0, 1)
sens = pd.DataFrame({"FN:FP ratio": ratios,
                     "best threshold (chosen)": [best_threshold(yv, p_va, r, 1)[0] for r in ratios],
                     "theory 1/(1+ratio)": 1 / (1 + ratios),
                     f"cost, chosen ({CHOSEN[0]})": [best_threshold(yv, p_va, r, 1)[1] for r in ratios],
                     f"cost, other ({OTHER[0]})": [best_threshold(yv, p_other, r, 1)[1] for r in ratios],
                     "cost, flag every wafer": [(1 - yv.mean())] * len(ratios)})
display(sens.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(ratios, sens["best threshold (chosen)"], "o-", color=STYLE[CHOSEN][0], label="best threshold on validation")
axes[0].plot(ratios, sens["theory 1/(1+ratio)"], color=MUTED, ls=":", label="theory for a calibrated model")
axes[0].set_xlabel("cost of a missed scratch, in unnecessary inspections"); axes[0].set_ylabel("threshold"); axes[0].set_title("Threshold vs cost ratio"); axes[0].legend()
axes[1].plot(ratios, sens[f"cost, chosen ({CHOSEN[0]})"], "o-", color=STYLE[CHOSEN][0], label=f"chosen: {CHOSEN[0]}")
axes[1].plot(ratios, sens[f"cost, other ({OTHER[0]})"], "s-", color=STYLE[OTHER][0], label=f"other: {OTHER[0]}")
axes[1].axvline(C_FN / C_FP, color=INK2, lw=1, ls="--"); axes[1].text(C_FN / C_FP, axes[1].get_ylim()[0], " our assumption", fontsize=8, color=INK2, va="bottom")
axes[1].set_xlabel("cost of a missed scratch, in unnecessary inspections"); axes[1].set_ylabel("cost per wafer at the best threshold"); axes[1].set_title("Evaluation loss vs cost ratio"); axes[1].legend()
plt.tight_layout(); savefig("06_cost_sensitivity.png"); plt.show()

,FN:FP ratio,best threshold (chosen),theory 1/(1+ratio),"cost, chosen (blackbox)","cost, other (glassbox)","cost, flag every wafer"
0,1,0.475,0.500,0.108,0.171,0.513
1,2,0.327,0.333,0.146,0.232,0.513
2,3,0.252,0.250,0.170,0.274,0.513
3,5,0.187,0.167,0.204,0.330,0.513
4,8,0.108,0.111,0.236,0.381,0.513
5,10,0.105,0.091,0.250,0.406,0.513
6,15,0.057,0.062,0.283,0.439,0.513
7,20,0.044,0.048,0.303,0.453,0.513


## 9. Interpretability: tree SHAP on the tree-based blackbox, global and local
We explain the **gradient-boosted trees trained with log loss** with tree SHAP. SHAP values are in log-odds: a value of +1 for a feature means that feature moved this wafer's log-odds of a scratch up by 1 from the average prediction.

The question we most want answered is whether the model looks at the scratch. If it does, the line features should lead. If the background features lead, the model is leaning on which other patterns tend to come with a scratch in this dataset, and that would not carry over to a fab.

We also put the glassbox's own explanation (its coefficients) next to the global SHAP ranking.

In [15]:
import shap
gbm = MODELS[("blackbox", "log loss")]
explainer = shap.TreeExplainer(gbm)
sv = np.asarray(explainer.shap_values(X_va))
if sv.ndim == 3: sv = sv[..., 1]
base_value = float(np.ravel(explainer.expected_value)[-1])
p_gbm = S_va[("blackbox", "log loss")]
logit = np.log(np.clip(p_gbm, 1e-9, 1 - 1e-9) / (1 - np.clip(p_gbm, 1e-9, 1 - 1e-9)))
print(f"additivity check: max |base + sum(SHAP) − model log-odds| = {np.abs(base_value + sv.sum(1) - logit).max():.2e}")

glob = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values()
lin = MODELS[("glassbox", "log loss")]
coef = pd.Series(lin[-1].coef_.ravel(), index=FEATURES)
rho = spearmanr(glob[FEATURES], coef.abs()[FEATURES]).correlation

seq = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#2a78d6", "#0d366b"])
fig, axes = plt.subplots(1, 3, figsize=(15, 6), gridspec_kw={"width_ratios": [1, 1.5, 1]})
axes[0].barh(glob.index, glob.values, color=[ORANGE if f in LINE else MUTED for f in glob.index], height=0.6); axes[0].grid(axis="y", visible=False)
axes[0].set_title("Global: mean |SHAP| (orange = line feature)"); axes[0].set_xlabel("mean |SHAP| (log-odds)")
rng = np.random.default_rng(SEED)
for i, f in enumerate(glob.index):
    j = FEATURES.index(f); v = X_va[f].rank(pct=True).values
    sc = axes[1].scatter(sv[:, j], i + rng.uniform(-0.3, 0.3, len(sv)), c=v, cmap=seq, s=5, alpha=0.7, linewidths=0)
axes[1].axvline(0, color=MUTED, lw=1); axes[1].set_yticks(range(len(glob))); axes[1].set_yticklabels(glob.index); axes[1].grid(axis="y", visible=False)
axes[1].set_title("Global: every validation wafer"); axes[1].set_xlabel("SHAP value (log-odds; right = towards Scratch)")
cb = fig.colorbar(sc, ax=axes[1], pad=0.01, shrink=0.8); cb.set_label("feature value (percentile)", fontsize=8); cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"])
c2 = coef[glob.index]
axes[2].barh(c2.index, c2.values, color=BLUE, height=0.6); axes[2].axvline(0, color=MUTED, lw=1); axes[2].grid(axis="y", visible=False)
axes[2].set_yticklabels([]); axes[2].set_title("Glassbox: L1 logistic coefficients"); axes[2].set_xlabel("coefficient per standard deviation (log-odds)")
plt.tight_layout(); savefig("07_shap_global.png"); plt.show()
print("global ranking (mean |SHAP|):", ", ".join(f"{f} {v:.2f}" for f, v in glob.iloc[::-1].items()))
print(f"features the L1 glassbox dropped: {list(coef.index[coef == 0]) or 'none'}")
print(f"rank agreement between mean |SHAP| and |glassbox coefficient| (Spearman): {rho:.2f}")
share_line = glob[LINE].sum() / glob.sum()
print(f"share of total mean |SHAP| carried by the {len(LINE)} line features: {share_line:.0%}; by the {len(BACKGROUND)} background features: {1 - share_line:.0%}")

additivity check: max |base + sum(SHAP) − model log-odds| = 5.26e-12


global ranking (mean |SHAP|): outer_ring_fail_rate 1.91, thin_fail_rate 1.37, line_score_top20 1.16, n_line_segments 0.74, edge_fail_rate 0.63, largest_cluster_radius 0.59, longest_thin_length 0.31, largest_cluster_share 0.26, longest_line_length 0.25, fail_rate 0.23, n_fail_clusters 0.22, middle_fail_rate 0.17, line_peak_contrast 0.14, angular_concentration 0.11, center_fail_rate 0.11, n_long_thin 0.03
features the L1 glassbox dropped: none
rank agreement between mean |SHAP| and |glassbox coefficient| (Spearman): 0.27
share of total mean |SHAP| carried by the 7 line features: 49%; by the 9 background features: 51%


In [16]:
top = glob.index[-1]; second = glob.index[-2]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, f in zip(axes, (top, second)):
    j = FEATURES.index(f)
    ax.scatter(X_va[f][yv == 0], sv[yv == 0, j], s=6, color=BLUE, alpha=0.5, linewidths=0, label="No scratch")
    ax.scatter(X_va[f][yv == 1], sv[yv == 1, j], s=6, color=ORANGE, alpha=0.5, linewidths=0, label="Scratch")
    ax.axhline(0, color=MUTED, lw=1); ax.set_xlabel(f); ax.set_ylabel("SHAP value (log-odds)"); ax.set_title(f"How {f} moves the prediction")
    if f in ("n_dies", "n_fail_clusters"): ax.set_xlim(X_va[f].min(), X_va[f].quantile(0.99))
axes[0].legend(markerscale=2.5)
plt.tight_layout(); savefig("08_shap_dependence.png"); plt.show()

### Local explanations, next to the wafer itself
Three validation wafers, chosen by rule (not by hand):
1. the scratched wafer the blackbox is **most sure** about
2. the unscratched wafer it is most sure about
3. its **costliest mistake**: the scratched wafer with the lowest predicted probability (a false negative)

Each row shows the wafer map and the eight largest SHAP values for that wafer.

In [17]:
z = np.load(MAPS); map_row = {r: i for i, r in enumerate(z["wafer_id"])}
wafer_cmap = ListedColormap(["#fcfcfb", "#cde2fb", ORANGE])        # no die, passing die, failing die

def wafer_map(row_id):
    m = z["maps"][map_row[row_id]]
    keep_r = np.flatnonzero((m != 255).any(1)); keep_c = np.flatnonzero((m != 255).any(0))
    return m[: keep_r.max() + 1, : keep_c.max() + 1]

pos, neg = np.flatnonzero(yv == 1), np.flatnonzero(yv == 0)
picks = [("Most confident scratch", pos[np.argmax(p_gbm[pos])]),
         ("Most confident no-scratch", neg[np.argmin(p_gbm[neg])]),
         ("Costliest mistake: a scratched wafer scored as clean", pos[np.argmin(p_gbm[pos])])]

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 3.3 * len(picks)), gridspec_kw={"width_ratios": [1, 2.6]})
for (title, i), (axm, axs) in zip(picks, axes):
    row = df.iloc[val_idx[i]]
    axm.imshow(wafer_map(row["wafer_id"]), cmap=wafer_cmap, vmin=0, vmax=2, interpolation="nearest"); axm.axis("off")
    axm.set_title(f"{title}\nlabel: {row['pattern']}", fontsize=9)
    contrib = pd.Series(sv[i], index=FEATURES); contrib = contrib.reindex(contrib.abs().sort_values().index[-8:])
    labels = [f"{f} = {X_va.iloc[i][f]:.3g}" for f in contrib.index]
    axs.barh(labels, contrib.values, color=[ORANGE if v > 0 else BLUE for v in contrib.values], height=0.6)
    axs.axvline(0, color=MUTED, lw=1); axs.grid(axis="y", visible=False)
    axs.set_title(f"base {base_value:+.2f} + all SHAP {sv[i].sum():+.2f} = log-odds {logit[i]:+.2f}  →  P(scratch) = {p_gbm[i]:.3f}   (8 largest shown)", fontsize=9)
    axs.set_xlabel("SHAP value (log-odds; orange pushes towards Scratch, blue towards no scratch)")
plt.tight_layout(); savefig("09_shap_local.png"); plt.show()
p_glass = S_va[("glassbox", "log loss")]
for title, i in picks:
    print(f"{title}: blackbox P = {p_gbm[i]:.3f}, glassbox P = {p_glass[i]:.3f}")

Most confident scratch: blackbox P = 1.000, glassbox P = 0.997
Most confident no-scratch: blackbox P = 0.000, glassbox P = 0.000
Costliest mistake: a scratched wafer scored as clean: blackbox P = 0.003, glassbox P = 0.552


**How far to trust these plots.** Every explanation rests on strong assumptions. Three apply here:
- **Correlated features.** Several features move together. SHAP splits credit among correlated features in a way that depends on how the trees happened to split, so the ranking *within* a correlated group isn't reliable. The split between the line group and the background group is.
- **SHAP explains the model, not the fab.** A large SHAP value says the model relies on a feature. It doesn't say that changing the feature would change the defect.
- **Part of the data is generated.** The dataset's authors balanced the classes with generated maps. A feature can matter because of how those maps were made.

## 10. Final check on the test split (used once)
Everything is frozen: the chosen model, its calibrator and its threshold. The test split has not been used for any decision in this notebook (see the disclosure in section 2).

In [18]:
yt = np.asarray(y_te)
p_te = calibrate(S_te[CHOSEN])
auc_te = roc_auc_score(yt, p_te)
lo, hi = boot_stats(yt, g_te, {"a": lambda idx: roc_auc_score(yt[idx], p_te[idx])})["a"]
flag = p_te >= T_CAL
tn, fp, fn, tp = confusion_matrix(yt, flag).ravel()
cost_te = eval_loss(yt, p_te, T_CAL)
c_lo, c_hi = np.percentile([eval_loss(yt[i], p_te[i], T_CAL) for i in wafer_bootstrap(g_te)], [2.5, 97.5])
final = pd.DataFrame({
    "validation": [roc_auc_score(yv, p_va), cost_cal, eval_loss(yv, p_va, 0.5), ece(yv, p_va), C_FP * (1 - yv.mean()), C_FN * yv.mean()],
    "test": [auc_te, cost_te, eval_loss(yt, p_te, 0.5), ece(yt, p_te), C_FP * (1 - yt.mean()), C_FN * yt.mean()],
}, index=["AUC", f"evaluation loss at threshold {T_CAL:.3f}", "evaluation loss at threshold 0.5", "ECE (calibrated)",
          "evaluation loss: flag every wafer", "evaluation loss: flag none"])
display(final.round(4))
print(f"test AUC 95% CI: {lo:.4f} to {hi:.4f}; test evaluation loss 95% CI: {c_lo:.4f} to {c_hi:.4f}")
print(f"test confusion at threshold {T_CAL:.3f}: TP {tp}, FN {fn}, FP {fp}, TN {tn} (recall {tp / (tp + fn):.1%}, precision {tp / (tp + fp):.1%}, flags {flag.mean():.1%} of wafers)")
print(f"for reference only (not used for any decision): test AUC of all four = "
      + ", ".join(f"{k[0]}/{k[1]} {roc_auc_score(yt, S_te[k]):.4f}" for k in MODELS))

summary = {"chosen_model": list(CHOSEN), "calibrator": CALIB, "threshold": round(T_CAL, 4), "cost_ratio_fn_fp": [C_FN, C_FP],
           "validation_auc": {f"{k[0]}|{k[1]}": round(float(roc_auc_score(yv, S_va[k])), 4) for k in MODELS},
           "test_auc_chosen": round(float(auc_te), 4), "test_eval_loss": round(float(cost_te), 4),
           "test_eval_loss_at_0.5": round(float(eval_loss(yt, p_te, 0.5)), 4), "configurations_tried": int(tuning["configurations tried"].sum())}
json.dump(summary, open("results_summary_scratch.json", "w"), indent=2); print(json.dumps(summary, indent=2))

,validation,test
AUC,0.9631,0.9605
evaluation loss at threshold 0.105,0.2499,0.2634
evaluation loss at threshold 0.5,0.6223,0.6496
ECE (calibrated),0.0086,0.0083
evaluation loss: flag every wafer,0.5134,0.5134
evaluation loss: flag none,4.8662,4.8662


test AUC 95% CI: 0.9567 to 0.9642; test evaluation loss 95% CI: 0.2417 to 0.2855
test confusion at threshold 0.105: TP 3541, FN 59, FP 1359, TN 2439 (recall 98.4%, precision 72.3%, flags 66.2% of wafers)
for reference only (not used for any decision): test AUC of all four = glassbox/L2 0.8861, glassbox/log loss 0.9074, blackbox/L2 0.9569, blackbox/log loss 0.9605
{
  "chosen_model": [
    "blackbox",
    "log loss"
  ],
  "calibrator": "Platt",
  "threshold": 0.1046,
  "cost_ratio_fn_fp": [
    10.0,
    1.0
  ],
  "validation_auc": {
    "glassbox|L2": 0.885,
    "glassbox|log loss": 0.9088,
    "blackbox|L2": 0.9578,
    "blackbox|log loss": 0.9631
  },
  "test_auc_chosen": 0.9605,
  "test_eval_loss": 0.2634,
  "test_eval_loss_at_0.5": 0.6496,
  "configurations_tried": 32
}


### Post-hoc audit of the model choice (reference only)
This is a check on our own decision, done after everything was frozen. The model we did **not** choose is put through the same steps (same calibrator rule, threshold set on validation) and scored on test. Nothing here changes the decision: picking a model after seeing its test score would make the test result meaningless.

In [19]:
raw_o = S_va[OTHER]
br = {"raw": brier_score_loss(yv, np.clip(raw_o, 0, 1)), "Platt": brier_score_loss(yv, CAL[OTHER]["Platt"](raw_o)),
      "isotonic": brier_score_loss(yv, CAL[OTHER]["isotonic"](raw_o))}
cal_o = min(br, key=br.get)
calibrate_o = (lambda s: np.clip(np.asarray(s, float), 0, 1)) if cal_o == "raw" else CAL[OTHER][cal_o]
t_o, cost_o_val = best_threshold(yv, calibrate_o(raw_o))
p_te_o = calibrate_o(S_te[OTHER])
cost_o_te = eval_loss(yt, p_te_o, t_o)
d_lo, d_hi = np.percentile([eval_loss(yt[i], p_te_o[i], t_o) - eval_loss(yt[i], p_te[i], T_CAL) for i in wafer_bootstrap(g_te)], [2.5, 97.5])
tn_o, fp_o, fn_o, tp_o = confusion_matrix(yt, p_te_o >= t_o).ravel()
audit = pd.DataFrame({
    f"chosen: {CHOSEN[0]}, {CHOSEN[1]}": [CALIB, T_CAL, cost_cal, cost_te, auc_te, accuracy_score(yt, p_te >= T_CAL), fn, fp],
    f"not chosen: {OTHER[0]}, {OTHER[1]}": [cal_o, t_o, cost_o_val, cost_o_te, roc_auc_score(yt, p_te_o), accuracy_score(yt, p_te_o >= t_o), fn_o, fp_o],
}, index=["calibrator", "threshold (set on validation)", "validation evaluation loss", "test evaluation loss", "test AUC", "test accuracy", "test FN", "test FP"])
display(audit)
print(f"test evaluation loss, not chosen − chosen = {cost_o_te - cost_te:+.4f} (95% CI {d_lo:+.4f} to {d_hi:+.4f})")

,"chosen: blackbox, log loss","not chosen: glassbox, log loss"
calibrator,Platt,Platt
threshold (set on validation),0.104641,0.119272
validation evaluation loss,0.249932,0.405921
test evaluation loss,0.26345,0.425656
test AUC,0.960538,0.907432
test accuracy,0.808327,0.698432
test FN,59,102
test FP,1359,2129


test evaluation loss, not chosen − chosen = +0.1622 (95% CI +0.1341 to +0.1909)


## 11. Findings, and what to flag

### What we found
1. **Line features make the scratch visible to a model.** On their own, the seven line features are the seven strongest single features (training AUC 0.63 to 0.73); no background feature exceeds 0.58.
2. **Log loss beats L2 for both model types.** Validation AUC is +0.024 higher for the glassbox (95% CI +0.020 to +0.028) and +0.005 for the blackbox (CI +0.004 to +0.007). The L2 models also put 10% (glassbox) and 24% (blackbox) of their scores outside 0–1, and the L2 glassbox is the worst calibrated of the four (ECE 0.077).
3. **We chose the blackbox.** Validation AUC is 0.963 for the boosted trees and 0.909 for the L1 logistic model (difference +0.054, CI +0.049 to +0.060), and the blackbox lowers the evaluation loss by 38% (0.250 vs 0.406 per wafer). That clears the 10% bar we set for giving up a readable model.
4. **Training AUC would have overstated it.** The blackbox scores 0.989 on its training data and 0.963 on validation.
5. **The threshold is where the cost ratio pays off.** The cost-minimising threshold is 0.117 on the raw score, not 0.5. That cuts the validation loss by 59% (0.613 → 0.250): 363 more scratched wafers caught (misses fall from 414 to 51) for 944 more false alarms. Accuracy falls from 89.1% to 81.2% while the cost falls by more than half.
6. **Calibration was good and got slightly better.** The blackbox's calibration error is 0.011 raw and 0.009 after Platt scaling, with AUC unchanged. On calibrated probabilities the best threshold is 0.105; decision theory gives 1/11 = 0.091, which costs almost the same (0.260 vs 0.250). For ratios from 1 to 20 the searched threshold stays within 0.03 of the formula.
7. **The choice holds for every cost ratio from 1 to 20.**
8. **Does the model look at the scratch? About half.** The line features carry 49% of the total SHAP attribution, with `thin_fail_rate` and `line_score_top20` second and third. The single largest feature is a background one, `outer_ring_fail_rate` (flag 1).
9. **The local explanations make physical sense.** The wafer the model is most sure about is a scratch-only wafer with a visible diagonal line, and four of its five largest contributions are line features. The costliest mistake is a wafer labelled Donut + Edge-Loc + Loc + Scratch, where large blobs cover most of the wafer and leave few thin structures: the model's line evidence is swamped. The linear model, for comparison, gave that wafer 0.55.
10. **Test result (used once).** AUC 0.961 (95% CI 0.957 to 0.964), evaluation loss 0.263 per wafer (CI 0.242 to 0.286), recall 98.4%, precision 72.3%, accuracy 80.8%. The loss is 49% below flagging every wafer (0.513) and 59% below using a 0.5 threshold (0.650). Validation and test agree (0.250 vs 0.263).
11. **The audit agrees with the choice.** The glassbox, put through the same steps, scores 0.426 on test, 0.162 worse than the blackbox (CI +0.134 to +0.191).

### Flags: where this project departs from the brief, or is weaker than it looks
| # | Flag | What we did about it |
|---|---|---|
| 1 | **Half of the model's attribution is background, and the top feature is `outer_ring_fail_rate`.** Some of that is legitimate context (a line stands out more on a quiet wafer). Some may be the dataset: which other patterns come with a Scratch label. | Reported openly, with the line / background split. We left position out of the features for the same reason. A fab would need to re-check this on its own data. |
| 2 | **The class balance is artificial.** 49% of wafers are scratched because the dataset's authors balanced the classes, partly with generated maps. | The cost per wafer is not a forecast for a fab. Section 8 shows the decision is stable across cost ratios; it does not show stability across scratch rates. |
| 3 | **"Avoid data cleaning."** We dropped 1,024 duplicate maps and withheld the die count, and the maps needed feature extraction. | Deliberate, with the evidence in section 1. |
| 4 | **The 10:1 cost ratio is our assumption.** | Stated up front; section 8 shows ratios 1–20. |
| 5 | **Two searches ended at the edge of their grids.** The blackbox chose the deepest trees offered (depth 8). The glassbox chose the weakest penalties, so it keeps all 16 features and its coefficients are hard to read one at a time. | We did not extend the grids after seeing validation. For the glassbox, cross-validated AUC is flat from C = 10 to 100, so a wider grid would not help it. |
| 6 | **This test split had been scored once before**, by an earlier version with a different feature set (section 2). | Disclosed. The features and grids here were designed on the training split. |
| 7 | **Random split.** The dataset has no lot or time information, and generated maps that resemble each other could sit in different splits. | Exact duplicates were removed. Near-duplicates cannot be ruled out. |
| 8 | **Labels and maps are partly synthetic**, so some "scratches" follow similar paths across wafers. | Position was left out of the features so the model cannot use it. |
| 9 | **Kaggle usability is 7.5 out of 10**, good but not at the top. | Noted. |

### Good habits we followed
- 32 configurations in total, tuned inside the training split. Validation answered one four-way question and set one threshold. Test was used once in this notebook.
- The feature set was fixed after three rounds on the training split, before any model was tuned.
- The decision rule was written down before the validation results.
- Every comparison has a bootstrap interval.
